# Optimal control by direct single shooting

Bryson-Denham for the dynamical system with control $u(t)$:

$$
\begin{aligned}
\min_{u(\cdot)}\quad & J = \tfrac12 \int_0^1 u(t)^2\,dt \\
\text{s.t.}\quad & \dot x = v, \\ \quad &\dot v = u, \\
& x(0) = 0,\ v(0) = \ \ \  1, \\ \qquad &x(1) =  0,\ v(1) = -1, \\
& x(t) \le \ell \quad \forall t \in [0, 1].
\end{aligned}
$$

In [ ]:
import jax
jax.config.update('jax_enable_x64', True)
import jax.numpy as jnp
import numpy as np
import diffrax
import matplotlib.pyplot as plt

import cottax.assemblies.jax as cx
from cottax.assemblies.jax import Start
flat = cx.flat

We're going to use the flat dictionary namespace and map these concepts.


First, we'll  declare the dynamics: 

In [ ]:
@flat.rate('x', 'v', initial={'x': 0.0, 'v': 1.0}, span='span', time='t',
           record={'x': 'x_s'}, times='t_s', kinks='t_knots')
def cart(x, v, u_t):                         
    return v, u_t

Where we additionally give dynamics, a span, a time variable (used for indexing in later functions) and switch on recording at times $t_s$ to save $x_s$.

The kinks indicate the discontinuous control variable for time-setting; the control is a linear interpolation from the $u$ array:

In [ ]:
@flat.function('u_t')                       # the control at the current time
def control(t, t_knots, u):
    return jnp.interp(t, t_knots, u)

The loss then  integrates it (linear so exact):

In [ ]:
@flat.function('J')                          # 1/2 int u^2 dt, exact for u linear between knots
def effort(u, t_knots):
    a, b, h = u[:-1], u[1:], jnp.diff(t_knots)
    return jnp.sum(h * (a * a + a * b + b * b)) / 6.0

and we then set a constraint on the final values and the path itself (on the knots)

In [ ]:
@flat.equality_constraint('hx', 'hv')                  # x(1) = 0, v(1) = -1
def terminal(x, v):
    return x, v + 1.0

@flat.inequality_constraint('path')                    # x(t_k) - l <= 0, one row per t_k
def path(x_s, l):
    return x_s - l

Then, we'll declare the optimisation:

In [ ]:
opt = flat.optimise(design={'u': (-20.0, 20.0)}, minimise='J', subject_to=(terminal, path))
declared = flat.declare(control, cart, effort, terminal, path, opt)

In [ ]:
cx.render_html(declared, blocks=True)

We can now immediately apply the MDF architecture, which in optimal control is just single-shooting:

In [ ]:

graph = declared + flat.mdf(opt) 
cx.render_html(graph, blocks=True)

Solving proceeds by assigning the drivers:

In [ ]:
drivers = (
    cx.ScipyMinimize(tol=1e-10, max_steps=500, jacobian=jax.jacfwd),
    cx.Diffrax(rtol=1e-10, atol=1e-10, adjoint=diffrax.ForwardMode()),
)

solver_graph = graph + flat.drive(*drivers)

In [ ]:
N, M, ELL = 21, 101, 1 / 9                   # control knots, path samples, the limit
t_knots = jnp.linspace(0.0, 1.0, N)
t_save  = jnp.linspace(0.0, 1.0, M)
inputs  = {'span': (0.0, 1.0), 't_s': t_save, 'l': ELL, 't_knots': t_knots}

out = flat.run(solver_graph, read_from={cx.Inputs: inputs, Start: {'u': jnp.zeros(N)}})

Comparing with the analytical solution:

In [ ]:
def analytic(t, l=ELL):
    s = jnp.minimum(t, 1 - t) / (3 * l)                   # mirrored entry/exit arcs
    on_arc = s < 1
    u = jnp.where(on_arc, -2 / (3 * l) * (1 - s), 0.0)
    x = jnp.where(on_arc, l * (1 - (1 - s) ** 3), l)
    return u, x

BLUE, ORANGE = '#2a78d6', '#eb6834'
t_fine = jnp.linspace(0, 1, 401)
u_ex, x_ex = analytic(t_fine)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.6))
a1.plot(t_knots, out['u'], color=BLUE, lw=2, marker='o', ms=4, label=f'cottax, N={N}')
a1.plot(t_fine, u_ex, color=ORANGE, lw=1.5, ls='--', label='analytic')
a1.set(xlabel='t', ylabel='u(t)', title='control')
a2.plot(t_save, out['x_s'], color=BLUE, lw=2, label='cottax')
a2.plot(t_fine, x_ex, color=ORANGE, lw=1.5, ls='--', label='analytic')
a2.axhline(ELL, color='0.55', lw=1, ls=':')
a2.annotate('ℓ', (0.02, ELL), xytext=(0, 4), textcoords='offset points', color='0.4')
a2.set(xlabel='t', ylabel='x(t)', title='state (path-constrained)')
for a in (a1, a2):
    a.grid(color='0.9', lw=0.8); a.spines[['top', 'right']].set_visible(False)
    a.legend(frameon=False)
fig.tight_layout()